# 02 · ML pipeline

Trains and compares every detector on the same building split, picks thresholds on the
validation buildings, evaluates on the unseen test buildings and saves the scorer bundle.

1. Expected consumption model (LightGBM regression on normal hours).
2. Detectors: rules, Random Forest, LightGBM (with and without weather, hybrid),
   Isolation Forest and the expected-consumption residual.
3. Product alarm (rule OR detector OR residual), alarm budgets, synthetic scenarios, events.

* Runtime: CPU. LightGBM gains nothing from a GPU or TPU at this size. The full run takes
  roughly 20-40 minutes on a standard Colab CPU runtime; `QUICK = True` takes a few minutes.
* Needs the processed tables; they are rebuilt automatically if missing.
* Same code as `python -m src.models.train`.

## 0. Setup

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO, BRANCH = "Ozgurisikdamar/enerji-anomali", "feature/data-ml"
USE_DRIVE = False  # keep data/ and models_out/ on Google Drive between Colab runtimes
IN_COLAB = "google.colab" in sys.modules


def colab_secret(name):
    from google.colab import userdata

    try:
        return userdata.get(name)
    except Exception:  # secret missing or notebook access not granted
        return None


if IN_COLAB:
    root = Path("/content/enerji-anomali")
    if not root.exists():
        url = f"https://github.com/{REPO}.git"
        token = colab_secret("GITHUB_TOKEN")
        auth_url = url.replace("https://", f"https://x-access-token:{token}@") if token else url
        clone = subprocess.run(
            ["git", "clone", "-q", "-b", BRANCH, auth_url, str(root)],
            capture_output=True,
            text=True,
        )
        if clone.returncode:
            raise RuntimeError("clone failed: check the GITHUB_TOKEN secret and repository access")
        subprocess.run(["git", "-C", str(root), "remote", "set-url", "origin", url], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(root / "requirements-colab.txt")],
        check=True,
    )
    if USE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
        store = Path("/content/drive/MyDrive/enerji-anomali")
        os.environ.update(
            DATA_DIR=str(store / "data"),
            MODEL_DIR=str(store / "models_out"),
            MLFLOW_DIR=str(store / "mlruns"),
        )
    if url := colab_secret("DATABASE_URL"):
        os.environ["DATABASE_URL"] = url
else:
    root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(root)
sys.path.insert(0, str(root))
print("repo:", root)

In [ ]:
import logging

import pandas as pd
from IPython.display import Image, Markdown, display

from src.config import get_settings
from src.models import tracking, train

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
settings = get_settings()
QUICK = False  # 20% of buildings, for a smoke test
TUNE = 0  # Optuna trials for the hybrid LightGBM, 0 = defaults
SOURCE = "files"  # "db" trains on every building in DATABASE_URL, with operator feedback as labels
TRACK = True  # log params, metrics, reports and the bundle to MLflow
PUBLISH = False  # section 6: replace the model the live scoring service uses

## 1. Build the training context

Loads the model frame (files or database) and builds the features: calendar, lags,
rolling statistics and weather.

In [ ]:
ctx = train.build_context(SOURCE, quick=QUICK)
labeled = ctx.labeled
print(f"rows {len(ctx.y):,} | labeled {labeled.sum():,} | anomaly rate {ctx.y[labeled].mean():.2%}")
ctx.frame.groupby("split")["building_id"].nunique()

## 2. Train every model

The expected-consumption model trains on normal hours only. Detectors train on train
buildings; thresholds are picked on validation buildings.

In [ ]:
trained = train.train_models(ctx, tune=TUNE)
display(pd.DataFrame([trained.regression]).round(3))
trained.table.round(4)

## 3. Build the scorer

The scorer bundles the expected model, the chosen detector and the thresholds. Training,
evaluation, the live scoring service and the demo all call `AnomalyScorer.score`.

In [ ]:
scorer = train.build_scorer(ctx, trained)
print("chosen detector:", train.chosen_detector(trained))
print("version:", scorer.version)
scorer.thresholds

## 4. Evaluate the product alarm on test buildings

Combined alarm quality, alarm budget table, synthetic scenario recall and costed events.

In [ ]:
ev = train.evaluate_product(ctx, trained, scorer, settings.tariff_tl_per_kwh)
display(pd.json_normalize(ev.combined).round(4))
display(ev.budgets.round(4))
ev.scenarios.round(4)

In [ ]:
ev.events.sort_values("est_cost_tl", ascending=False).head(10)

## 5. Save the model and the reports

Writes the scorer bundle to `models_out/`, the results to `reports/ml/` and logs the run to
MLflow: `MLFLOW_TRACKING_URI` if set, otherwise a SQLite store in `mlruns/` (on Drive when
`USE_DRIVE`). Open it later with `mlflow ui --backend-store-uri sqlite:///mlruns/mlflow.db`.

In [ ]:
with tracking.run("notebook", enabled=TRACK) as tracker:
    summary, model_dir, out = train.save_run(ctx, trained, scorer, ev, tracker, QUICK, TUNE)
display(Markdown((out / "results.md").read_text()))

In [ ]:
for path in sorted((out / "figures").glob("*.png")):
    display(Image(str(path)))

## 6. Optional: publish to the shared database

Stores the model in `model_artifacts` and the test predictions and events with origin
`batch`, so the API and the frontend show them. The scoring service switches to this model
on its next cycle, so set `PUBLISH = True` only for a full run you want live. Needs a
`DATABASE_URL` with write access.

In [ ]:
from src.models.publish import publish_results

if not PUBLISH:
    print("PUBLISH is off, skipping")
elif url := settings.database_url:
    publish_results(url, model_dir, ev.test_scored, ev.events)
    print("published", scorer.version)
else:
    print("DATABASE_URL not set, skipping")

## 7. Keep the results

Commit `reports/ml` to the repository. Never commit `models_out/`; publish the model to the
database or keep the zip.

In [ ]:
if IN_COLAB:
    import shutil

    from google.colab import files

    files.download(shutil.make_archive("/content/ml_results", "zip", settings.reports_dir / "ml"))
    files.download(shutil.make_archive("/content/model", "zip", model_dir))